# AdaptCLIP MVTec Result Table

MVTec 전체 클래스에 대해 AdaptCLIP image-level AUROC와 top score를 저장합니다.


In [ ]:
from pathlib import Path
import sys

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "models").is_dir() and (path / "datasets").is_dir() and (path / "utils").is_dir():
            return path
    raise RuntimeError("Repository root was not found from the current working directory.")

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) in sys.path:
    sys.path.remove(str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT))

print("repo root:", REPO_ROOT)


In [ ]:
import csv
import os

import pandas as pd

import config_mv
from datasets.mvtec import MyData
from models.adaptclip import AdaptCLIP
from utils.metrics import get_image_auc


In [ ]:
import torch

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(device)


In [ ]:
categories = [
    "bottle", "cable", "capsule", "carpet", "grid", "hazelnut",
    "leather", "metal_nut", "pill", "screw", "tile", "toothbrush",
    "transistor", "wood", "zipper",
]

results = []

for name in categories:
    train_data = MyData(
        name,
        phase="train",
        batch_size=config_mv.BATCH_SIZE,
        shuffle=False,
        limit=config_mv.TRAIN_LIMIT,
    )

    test_data = MyData(
        name,
        phase="test",
        batch_size=config_mv.BATCH_SIZE,
        shuffle=False,
        limit=config_mv.TEST_LIMIT,
        limit_per_class=config_mv.TEST_LIMIT_PER_CLASS,
    )

    adaptclip = AdaptCLIP(
        category=name,
        device=device,
        checkpoint_path="mvtec",
        img_resize=336,
        img_cropsize=336,
        batch_size=config_mv.BATCH_SIZE,
        use_prompt_query=True,
    )
    adaptclip.fit(train_data)

    scores = []
    for img, label in test_data:
        score, _ = adaptclip.predict(img)
        scores.append(score.item())

    image_auc = get_image_auc(test_data, adaptclip)
    top_score = max(scores) if scores else float("nan")

    results.append([
        name,
        config_mv.TRAIN_LIMIT,
        "mvtec",
        round(image_auc, 4),
        round(top_score, 4),
    ])
    print(name, "AUROC", image_auc)

results = sorted(results, key=lambda x: x[3], reverse=True)

file_path = "adaptclip_mvtec_results.csv"
with open(file_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["category", "train_limit", "checkpoint", "auc", "top_score"])
    writer.writerows(results)

file_path


In [ ]:
df = pd.read_csv("adaptclip_mvtec_results.csv")
df
